In [ ]:
import numpy as np
from PIL import Image
import os

In [ ]:
def transformation(img, M, fill=255):
  H, W = img.shape[:2]
  C = img.shape[2]
  cx, cy = (W - 1) / 2.0, (H - 1) / 2.0
  ys, xs = np.mgrid[0:H, 0:W]
  P = np.stack([xs.ravel() - cx, ys.ravel() - cy], 1)
  Q = P @ M.T
  qx = np.round(Q[:, 0]).astype(int)
  qy = np.round(Q[:, 1]).astype(int)
  qx -= qx.min()
  qy -= qy.min()
  out = np.full((qy.max() + 2, qx.max() + 2, C), fill, dtype=img.dtype)
  src = img.reshape(-1, C)
  out[qy, qx] = src
  out[qy, qx + 1] = src
  out[qy + 1, qx] = src
  return out

In [ ]:
def scaling(bx, by):
    return np.array([[bx, 0.0], [0.0, by]])

def shear(theta):
  return np.array([[1.0, np.tan(np.deg2rad(theta))], [0.0, 1.0]])

def rotation(phi):
  p = np.deg2rad(phi)
  return np.array([[np.cos(p), -np.sin(p)], [np.sin(p), np.cos(p)]])

def reflection(axis):
  if axis == "x":
    return np.array([[1.0, 0.0], [0.0, -1.0]])
  else:
    return np.array([[-1.0, 0.0], [0.0, 1.0]])

In [ ]:
def save_image(arr, path):
  Image.fromarray(arr.astype(np.uint8)).save(path)

In [ ]:
path = "test1.png"
img = np.asarray(Image.open(path).convert("RGB"))
name = path.rsplit(".", 1)[0]
out_dir = f"{name}_out"
os.makedirs(out_dir)

for bx, by in [(-1.75, -1.75), (-1.5, 1), (1, -1.5), (1.75, 1.75), (1.5, 1), (1, 1.75)]:
  save_image(transformation(img, scaling(bx, by)), f"{out_dir}/{name}_scale_{bx}_{by}.png")

for theta in [60, 45, 30, 15, -15, -45, -60]:
  save_image(transformation(img, shear(theta)), f"{out_dir}/{name}_shear_{theta}.png")

for phi in [60, 45, 30, 15, -15, -45, -60]:
  save_image(transformation(img, rotation(phi)), f"{out_dir}/{name}_rotation_{phi}.png")

for axis in ["x", "y"]:
  save_image(transformation(img, reflection(axis)), f"{out_dir}/{name}_reflection_{axis}.png")


In [ ]:
import shutil
shutil.make_archive("test1_out", "zip", "test1_out")
shutil.make_archive("test2_out", "zip", "test2_out")

'/content/test2_out.zip'